# Full Teacher Dataset Generation

Bu notebook yalnızca **frozen 4,000-bank environment** dosyasından production
teacher dataset üretir.

Locked production ayarları:

\[
4000\ bank \times 32W \times 512Z,\qquad N_{MC}=64,000
\]

Her 10 bank bir Parquet shard olur. Shard önce `/content` local SSD'de tamamlanır,
sonra Drive'a kopyalanır. `manifest.json` sayesinde aynı komut tekrar
çalıştırıldığında tamamlanmış shard'lar atlanır.

## 1. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. GitHub repo clone / pull

In [ ]:
from google.colab import userdata
from pathlib import Path
import os, subprocess, tempfile

token = userdata.get("GITHUB_TOKEN")
repo_url = "https://github.com/OsmanMertYilmaz/Phy_Informed_MIMO_RIS_Env.git"
repo_dir = Path("/content/Phy_Informed_MIMO_RIS_Env")

askpass = tempfile.NamedTemporaryFile(mode="w", delete=False, suffix=".sh")
askpass.write("""#!/bin/sh
case "$1" in
    *Username*) echo "OsmanMertYilmaz" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")
askpass.close()
os.chmod(askpass.name, 0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = askpass.name
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    if (repo_dir / ".git").exists():
        print("Repo var -> git pull")
        subprocess.run(
            ["git","-C",str(repo_dir),"pull","--ff-only"],
            env=env, check=True
        )
    else:
        print("Repo yok -> git clone")
        subprocess.run(
            ["git","clone",repo_url,str(repo_dir)],
            env=env, check=True
        )
finally:
    os.remove(askpass.name)

print("GitHub sync tamamlandı.")

## 3. Package + tests

In [ ]:
%cd /content/Phy_Informed_MIMO_RIS_Env
!pip install -e .
!pytest -q

## 4. GPU ve frozen environment kontrolü

In [ ]:
from pathlib import Path
import pandas as pd
import torch

ENV = Path(
    "/content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/"
    "environments/environments_4000.csv"
)
OUT = Path(
    "/content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/"
    "teacher_dataset"
)

assert ENV.exists(), ENV
assert torch.cuda.is_available(), "CUDA aktif değil."

D = pd.read_csv(ENV)

print("GPU       :", torch.cuda.get_device_name(0))
print("Banks     :", len(D))
print("Splits    :")
print(D["splitID"].value_counts())
print("Environment:", ENV)
print("Output     :", OUT)

assert len(D) == 4000

## 5. Önce sadece status

Bu hücre hesaplama başlatmaz. Mevcut `manifest.json` ve shard'lara bakar.
İlk çalıştırmada `%0` görmen normal.

In [ ]:
%cd /content/Phy_Informed_MIMO_RIS_Env

!python scripts/generate_teacher_dataset.py \
    --environments /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments/environments_4000.csv \
    --output-root /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/teacher_dataset \
    --local-root /content/ris_teacher_work \
    --n-mc 64000 \
    --k-w 32 \
    --banks-per-shard 10 \
    --mc-chunk 256 \
    --w-chunk 4 \
    --z-chunk 64 \
    --status-only

## 6A. İlk production shard — önce bunu çalıştır

Bu hücre yalnızca **10 bank = 1 shard** üretir:

\[
10\times32\times512 = 163{,}840\ \text{satır}.
\]

Writer, Parquet, manifest ve resume zincirini gerçek production formatında
doğrulamak için önce bunu çalıştır. `SHARD PASS` gördükten sonra bana çıktıyı
gönder; sonra 6B'deki full run'a geç.

In [ ]:
%cd /content/Phy_Informed_MIMO_RIS_Env

!python scripts/generate_teacher_dataset.py \
    --environments /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments/environments_4000.csv \
    --output-root /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/teacher_dataset \
    --local-root /content/ris_teacher_work \
    --n-mc 64000 \
    --k-w 32 \
    --banks-per-shard 10 \
    --mc-chunk 256 \
    --w-chunk 4 \
    --z-chunk 64 \
    --max-banks 10

## 6B. Full production generation — 6A PASS sonrasında

Aynı komut `--max-banks` olmadan çalışır. İlk shard manifest'te tamamlandıysa
onu atlar ve kalan banklardan devam eder.

Colab koparsa **aynı hücreyi yeniden çalıştır**.

In [ ]:
%cd /content/Phy_Informed_MIMO_RIS_Env

!python scripts/generate_teacher_dataset.py \
    --environments /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments/environments_4000.csv \
    --output-root /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/teacher_dataset \
    --local-root /content/ris_teacher_work \
    --n-mc 64000 \
    --k-w 32 \
    --banks-per-shard 10 \
    --mc-chunk 256 \
    --w-chunk 4 \
    --z-chunk 64

## 7. İlerlemeyi kontrol et

In [ ]:
%cd /content/Phy_Informed_MIMO_RIS_Env

!python scripts/generate_teacher_dataset.py \
    --environments /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments/environments_4000.csv \
    --output-root /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/teacher_dataset \
    --local-root /content/ris_teacher_work \
    --n-mc 64000 \
    --k-w 32 \
    --banks-per-shard 10 \
    --mc-chunk 256 \
    --w-chunk 4 \
    --z-chunk 64 \
    --status-only

## 8. Manifest özeti

In [ ]:
import json
from pathlib import Path

MANIFEST = Path(
    "/content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/"
    "teacher_dataset/manifest.json"
)

m = json.loads(MANIFEST.read_text())
print("Completed banks :", len(m["completed_banks"]), "/", m["total_banks"])
print("Completed shards:", len(m["completed_shards"]))

if m["completed_shards"]:
    print("\nLast shard:")
    print(json.dumps(m["completed_shards"][-1], indent=2)[:5000])